## ఎంబెడ్డింగ్స్

మన పుర్వ అధ్యయనంలో, మేము పొడవు `vocab_size` ఉన్న అధిక-మానం బ్యాగ్-ఆఫ్-వర్డ్స్ వెక్టర్లపై పనిచేశాము, మరియు మేము తక్కువ-మానం స్థానిక ప్రాతినిధ్యం వెక్టర్ల నుంచి అరుదైన వన్-హాట్ ప్రాతినిధ్యం లోకి స్పష్టంగా కన్వర్ట్ చేసుకున్నాము. ఈ వన్-హాట్ ప్రాతినిధ్యం స్మృతి-సంక్షమం కాదు, అదనంగా ప్రతి పదం ఇతర పదాల నుంచి స్వతంత్రంగా పరిగణించబడతాయి, అంటే వన్-హాట్ ఎన్కోడ్ అయిన వెక్టర్లు పదాల మధ్య ఎటువంటి సార్వభౌమ సార్ధకతను వ్యక్తం చేయవు.

ఈ యూనిట్‌లో, మనం **News AG** డేటాసెట్‌ను కొనసాగించి పరిశీలిస్తాం. మొదలు పెట్టడానికి, డేటాను లోడ్ చేసి ముందటి నోట్‌బుక్ నుండి కొంత నిర్వచనలు తీసుకుందాం.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## ఎంబెడ్డింగ్ అంటే ఏమిటి?

**ఎంబెడ్డింగ్** యొక్క ఆలోచన అనేది పదాలను తక్కువ-డైమెన్షనల్ డెన్స్ వెక్టార్లతో ప్రాతినిధ్యం చేయడం, అది ఎలా నచ్చిన అభిప్రాయాన్ని ప్రతిబింబిస్తుంది. తర్వాత మనం అర్థవంతమైన పద ఎంబెడ్డింగ్‌లు ఎలా నిర్మించాలో చర్చించబోతున్నాం, అయితే ప్రస్తుతానికి ఎంబెడ్డింగ్‌లను ఒక పద వెక్టర్ డైమెన్షనాలిటీ తగ్గించే సాధనంగా భావిద్దాం.

అందుకే, ఎంబెడ్డింగ్ లేయర్ ఒక పదాన్ని ఇన్పుట్‌గా తీసుకొని, నిర్దిష్టమైన `embedding_size` యొక్క అవుట్పుట్ వెక్టర్‌ను ఉత్పత్తి చేస్తుంది. ఒక అర్థంలో, ఇది `Linear` లేయర్‌కు చాలా సమానంగా ఉంటుంది, కానీ ఒక-హాట్ ఎన్‌కోడెడ్ వెక్టర్ తీసుకోవడం మర్చిపోతూ, పద సంఖ్యను ఇన్పుట్‌గా తీసుకోవడం చేస్తుంది.

మన నెట్‌వర్క్‌లో మొదటి లేయర్‌గా ఎంబెడ్డింగ్ లేయర్ ఉపయోగించి, మనం బాగ్-ఆఫ్-వర్డ్స్ నుండి **ఎంబెడ్డింగ్ బాగ్** మోడల్‌కు మార్చవచ్చు, ఇక్కడ మనం మొదట మన టెక్స్ట్‌లోని ప్రతి పదాన్ని తమకు తగిన ఎంబెడ్డింగ్‌గా మార్చి, ఆ ఎంబెడ్డింగ్‌లపై агрегేట్ ఫంక్షన్‌ను లెక్కింపు చేస్తాము, ఉదాహరణకు `sum`, `average` లేదా `max`.

![ఐదు క్రమబద్ధ పదాలు కోసం ఎంబెడ్డింగ్ క్లాసిఫైయర్ చూపించబడుతోంది.](../../../../../translated_images/te/embedding-classifier-example.b77f021a7ee67eee.webp)

మా క్లాసిఫైయర్ న్యూరల్ నెట్‌వర్క్ ఎంబెడ్డింగ్ లేయర్‌తో ప్రారంభమవుతుంది, తర్వాత అగ్రిగేషన్ లేయర్, మరియు దాని పై లినియర్ క్లాసిఫైయర్ ఉంటుంది:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### మార్పిడి గల వరుస పరిమాణం నిర్వహణ

ఈ معماری ఫలితంగా, మా నెట్వర్క్ కు మినీబాచ్ లను నిర్దిష్టంగా సృష్టించాల్సి ఉంటుంది. గత యూనిట్ లో, బాగ్-ఆఫ్-వర్డ్స్ ఉపయోగించినప్పుడు, మినీబాచ్ లోని అన్ని BoW టెన్సార్లు సమాన పరిమాణం కలిగి ఉండేవి `vocab_size`, మా వచన వరుస యొక్క నిజమైన పొడవు ఏమైనా సంబంధం లేకుండా. ఒకసారి మేము వర్డ్ ఎంబెడ్డింగ్స్ వైపు వెళితే, ప్రతి వచన నమూనాలో పదాల సంఖ్య మార్పిడి అయి ఉండేది, మరియు ఆ నమూనాలను మినీబాచ్ లుగా కలుపుతేము, కొన్ని ప్యాడింగ్ వర్తించాల్సి ఉంటుంది.

ఇది datasource కి `collate_fn` ఫంక్షన్ అందించే అదే పద్ధతితో చేయవచ్చు:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### ఎంబెడింగ్ క్లాసిఫయర్ శిక్షణ

ఇప్పుడు మనకు సరైన డేటాలోడర్ నిర్వచించబడినందువల్ల, మునుపటి యూనిట్‌లో నిర్వచించుకున్న శిక్షణ ఫంక్షన్ ఉపయోగించి మodel ను శిక్షణ ఇవ్వవచ్చు:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **గమనిక**: సమయ కారణాలతో మేము ఇక్కడ 25k రికార్డులు మాత్రమే శిక్షణ ఇస్తున్నాము (ఒక పూర్తి యుపోచ్ కన్నా తక్కువ), కానీ మీరు శిక్షణను కొనసాగించవచ్చు, అనేక యుపోచ్లకు శిక్షణ ఇస్తే ఫంక్షన్ రాయవచ్చు, మరియు ఉన్నత సజావుల కోసం లెర్నింగ్ రేట్ పారామీటర్‌తో ప్రయోగాలు చేయవచ్చు. మీరు సుమారు 90% సజావుకు చేరువ అవ్వగలరు.


### ఎంబెడింగ్ బ్యాగ్ లేయర్ మరియు వేరియబుల్-లెంత్ సీక్వెన్స్ ప్రాతినిధ్యం

మునుపటి ఆర్కిటెక్చర్‌లో, మినీబాచ్లో పెట్టడానికి అన్ని సీక్వెన్స్‌లను ఒకే పొడవు వరకు ప్యాడ్ చేయాలి. ఇది వేరియబుల్ పొడవు సీక్వెన్స్‌లను ప్రాతినిధ్యం చేసే అత్యంత సమర్థవంతమైన మార్గం కాదు - మరో విధానం **ఆఫ్సెట్** వెక్టర్‌ను ఉపయోగించడం, ఇది ఒక పెద్ద వెక్టర్‌లో నిల్వ చేయబడిన అన్ని సీక్వెన్స్‌ల ఆఫ్సెట్లను కలిగి ఉంటుంది.

![ఆఫ్సెట్ సీక్వెన్స్ ప్రాతినిధ్యం చూపించే చిత్రం](../../../../../translated_images/te/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **గమనిక**: పై చిత్రంలో, మనం అక్షరాల సీక్వెన్స్‌ను చూపిస్తున్నాము, కానీ మన ఉదాహరణలో మేము పదాల సీక్వెన్స్‌లతో పని చేస్తున్నాము. అయితే, ఆఫ్సెట్ వెక్టర్‌తో సీక్వెన్స్‌లను ప్రాతినిధ్యం చేసే సాధారణ సిద్ధాంతం అదే ఉంటుంది.

ఆఫ్సెట్ ప్రాతినిధ్యంతో పని చేసే కోసం, మేము [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) లేయర్‌ను ఉపయోగిస్తాము. ఇది `Embedding`కు సమానమైనది, కానీ ఇది కంటెంట్ వెక్టర్ మరియు ఆఫ్సెట్ వెక్టర్‌ను ఇన్‌పుట్‌గా తీసుకుంటుంది, అలాగే ఇది సరాసరి లేయర్‌ను కలిగి ఉంటుంది, అది `mean`, `sum` లేదా `max` కావచ్చు.

ఇక్కడ `EmbeddingBag` ఉపయోగించే సవరించిన నెట్‌వర్క్ ఉంది:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

dataset ను training కోసం సిద్ధం చేయడానికి, offset vector ను సిద్దం చేసే conversion function ను అందించాల్సి ఉంటుంది:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

గమనించండి, అన్ని పూర్వ ఉదాహరణల వల్ల వేరు, మన నెట్‌వర్క్ ఇప్పుడు రెండు పరామితులను స్వీకరిస్తుంది: డేటా వెక్టర్ మరియు ఆఫ్సెట్ వెక్టర్, ఇవి వేర్వేరు పరిమాణాలుగా ఉంటాయి. ఇదే విధంగా, మన డేటా లోడర్ కూడా మాకు 2 కాకుండా 3 విలువలను అందిస్తుంది: టెక్ట్స్ మరియు ఆఫ్సెట్ వెక్టర్లు రెండూ లక్షణాలుగా అందిస్తారు. కనుక, మనం మన ట్రైనింగ్ ఫంక్షన్‌ను కొంచెం సరిచేయాల్సివస్తుంది అవీ చూసేందుకు:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## సేమాంటిక్ ఎంపెడ్డింగ్స్: Word2Vec

మునుపటి ఉదాహరణలో, మోడల్ ఎంపెడ్డింగ్ లేయర్ పదాలను వెక్టర్ ప్రాతినిధ్యం కోసం మ్యాప్ చేయడం నేర్చుకుంది, అయితే ఈ ప్రాతినిధ్యం ఎక్కువ సేమాంటికల్ అర్థం కలిగినది కాదు. ఇటువంటి వెక్టర్ ప్రాతినిధ్యం నేర్చుకోవడం బాగుంటుంది, అంటే సమాన పదాలు లేదా సమానార్ధకాలు కొన్ని వెక్టర్ దూరం (ఉదా. యూక్లిడియన్ దూరం) పరంగా ఒకరికి మరోటి దగ్గరగా ఉండే వెక్ట్‌ర్లకు అనుగుణంగా ఉంటాయి.

దీని కోసం, మనం ఒక నిర్దిష్ట విధంగా పెద్ద పరిమాణంలో టెక్స్ట్ సేకరణపై మన ఎంపెడ్డింగ్ మోడల్‌ను ప్రీ-ట్రెయిన్ చేయాలి. సేమాంటిక్ ఎంపెడ్డింగ్స్ నేర్పే మొదటి విధానాల్లో ఒకటి [Word2Vec](https://en.wikipedia.org/wiki/Word2vec) అంటారు. ఇది పదాలకు పంపిణీ ప్రాతినిధ్యం ఉత్పత్తి చేయడానికి ఉపయోగించే రెండు ప్రధాన معماريలు ఆధారంగా ఉంటుంది:

 - **సतత బాగ్-ఆఫ్-వర్డ్‌സ്** (CBoW) — ఈ معماريలో, మోడల్‌ను చుట్టుపక్కల ఉన్న సందర్భం నుండి ఒక పదాన్ని ఊహించేందుకు ట్రెయిన్ చేస్తారు. nగ్రామ్ $(W_{-2},W_{-1},W_0,W_1,W_2)$ ఇవ్వబడినప్పుడు, మోడల్ లక్ష్యం $(W_{-2},W_{-1},W_1,W_2)$ నుండి $W_0$ని ఊహించడమే.
 - **సతత స్కిప్-గ్రామ్** CBoW కి విరుద్ధం. మోడల్ ప్రస్తుత పదాన్ని ఊహించడానికి చుట్టుపక్కల ఉన్న విండో సందర్భ పదాలను ఉపయోగిస్తుంది.

CBoW వేగంగా పనిచేస్తుంది, స్కిప్-గ్రామ్ మందగిస్తది, కాని అరుదైన పదాలను ప్రాతినిధ్యం చేయడంలో మెరుగుగా పని చేస్తుంది.

![పదాలను వెక్టర్‌లుగా మార్చేందుకు CBoW మరియు స్కిప్-గ్రామ్ ఆల్గోరిదమ్లను చూపించే చిత్రం.](../../../../../translated_images/te/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

గూగుల్ న్యూస్ డేటాసెట్‌పై ప్రీ-ట్రెయిన్ చేసిన word2vec ఎంపెడ్డింగ్‌తో ప్రయోగం చేయడానికి, మనం **gensim** లైబ్రరీ ఉపయోగించవచ్చు. కింది ఉదాహరణలో 'neural' కు అత్యంత సాదృశ్యమున్న పదాలను మనం కనుగొంటాం

> **గమనిక:** మీరు మొదటిసారి word vectors సృష్టించినప్పుడు, వాటిని డౌన్‌లోడ్ చేయడానికి కొంత సమయం పడుతుంది!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


మేము కృత్రిమ వర్గీకరణ మోడల్ శిక్షణలో ఉపయోగించడానికి ఈ పదం నుండి వెక్టర్ ఎంబెడింగ్‌ల‌ను కూడా లెక్కించవచ్చు (స్పష్టత కోసం వెక్టర్ యొక్క మొదటి 20 భాగాలు మాత్రమే చూపించాము):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

సెమాంటికల్ ఎంబెడ్డింగ్స్ గురించి గొప్ప విషయం ఏమిటంటే, మీరు వెక్టర్ ఎన్‌కోడింగ్‌ను మార్చి సెమాంటిక్స్‌ని మార్చవచ్చు. ఉదాహరణకు, మేము ఒక పదాన్ని కనుగొనమని వ్యక్తం చేయవచ్చు, దాని వెక్టర్ ప్రాతినిధ్యం పదాలు *king* మరియు *woman*కి όσο దగ్గరగా ఉండాలి, మరియు పదం *man* నుండి όσο దూరంగా ఉండాలి:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

CBoW మరియు Skip-Grams రెండుఇ “అంచనా వేయు” ఎంబెడ్డింగ్లు, అవి కేవలం స్థానిక పరిసరాలను మాత్రమే పరిగణలోకి తీసుకుంటాయి. Word2Vec గ్లోబల్ కాంటెక్స్ట్ ను ఉపయోగించదు.

**FastText**, Word2Vec పైన నిర్మించబడింది, ప్రతి మాట మరియు ప్రతి మాటలోని అక్షర n-గ్రామ్‌ల కోసం వెక్టర్ ప్రాతినిధ్యాలను నేర్చుకుంటుంది. ప్రాతినిధ్య విలువలు తరబడి ఒకటే వెక్టర్‌గా సగటుపరుస్తారు ప్రతీ శిక్షణ దశలో. ఇది ప్రి-ట్రైనింగ్‌కు అనేక అదనపు గణనలను కలిగి వస్తున్నప్పటికీ, మాటల ఎంబెడ్డింగ్లకు ఉప-మాట సమాచారం ఎన్‌కోడ్ చేయడం సులభమవుతుంది.

మరో విధానం, **GloVe**, సహ-ఘటనా మ్యాట్రిక్స్ భావనను ఉపయోగించి, సహ-ఘటనా మ్యాట్రిక్స్ ను మరింత వ్యక్తీకరించదగిన మరియు అసారలైన మాట వెక్టార్లుగా విచ్ఛిన్నం చేయడానికి న్యూరల్ పద్ధతులను ఉపయోగిస్తుంది.

gensim అనేక రకాల వర్డ్ ఎంబెడ్డింగ్ మోడళ్లను మద్దతుతోందున, మీరు FastText మరియు GloVe కు ఎంబెడ్డింగ్లను మార్చి ఉదాహరణతో ఆడవచ్చు.


## PyTorch లో ప్రీ-శిక్షణ పొందిన ఎంబెడ్డింగ్స్ ఉపయోగించడం

మేము పై ఉదాహరణను మార్చి మా ఎంబెడ్డింగ్ లేయర్‌లో ఉన్న మ్యాట్రిక్స్‌ను Word2Vec వంటి సాందర్భిక ఎంబెడ్డింగ్స్‌తో ముందుగా పూరించవచ్చు. ప్రీ-శిక్షణ పొందిన ఎంబెడ్డింగ్ మరియు మా పాఠ్య సేకరణ యొక్క శబ్దకోశాలు సాధారణంగా సరిపోదని మనం భావించాలి, కాబట్టి లేని పదాల కోసం వెయిట్లు యాదృచ్ఛిక విలువలతో ప్రారంభిస్తాము:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


ఇప్పుడు మన మోడల్ను శిక్షణిద్దాం. మోడల్ శిక్షణకు అవసరమయ్యే సమయం గత ఉదాహరణతో పోలిస్తే గణనీయంగా ఎక్కువగా ఉందని గమనించండి, ఎందుకంటే ఎంబెడ్డింగ్ లేయర్ పరిమాణం ఎక్కువగా ఉండటం వలన, మిగతా పారామితుల సంఖ్య కూడా చాలా ఎక్కువ గలదు. అంతేగాక, ఈ కారణంగా, మేము ఓవర్‌ఫిట్టింగ్ నివారించాలనుకుంటే మోడల్ ని మరిన్ని ఉదాహరణలపై శిక్షణ ఇవ్వాల్సి రావచ్చు.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

మన కేసులో, నోట్లు చాలా విభిన్నమైన పదసామగ్రి కావడం వల్ల ఖచ్చితత్వంలో పెద్ద పెరుగుదల కనిపించడం లేదు. 
విభిన్న పదసామగ్రి సమస్యను అధిగమించడానికి, మేము క్రింది పరిష్కారాల్లో ఒకదాన్ని ఉపయోగించవచ్చు:
* మన పదసామగ్రిపై word2vec మోడల్‌ను మళ్ళీ శిక్షణ ఇవ్వండి
* ముందుగా శిక్షణ పొందిన word2vec మోడల్ నుండి వచ్చే పదసామగ్రితో మన డేటాసెట్‌ను లోడ్ చేయండి. డేటాసెట్ లోడ్ సమయంలో ఉపయోగించే పదసామగ్రి పేర్కొనవచ్చు.

తర్వాతి విధానం సులభంగా కనిపిస్తుంది, ముఖ్యంగా PyTorch `torchtext` ఫ్రేమ్‌వర్క్‌లో ఎంబెడింగ్స్‌కు ఇంటిగ్రేటెడ్ మద్దతు ఉండటం వలన. ఉదాహరణకు, మనం కింది విధంగా GloVe ఆధారిత పదసామగ్రి సృష్టించవచ్చు:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary has the following basic operations:
* `vocab.stoi` dictionary allows us to convert word into its dictionary index
* `vocab.itos` does the opposite - converts number into word
* `vocab.vectors` is the array of embedding vectors, so to get the embedding of a word `s` we need to use `vocab.vectors[vocab.stoi[s]]`

Here is the example of manipulating embeddings to demonstrate the equation **kind-man+woman = queen** (I had to tweak the coefficient a bit to make it work):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

ఆ ఎంబెడ్డింగ్లను ఉపయోగించి క్లాసిఫయర్‌ను శిక్షణ ఇవ్వడానికి, ముందుగా మన డేటాసెట్‌ను GloVe పదసంపద ఉపయోగించి ఎంకోడ్ చేయాలి:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

మేము పైగా చూసినట్లుగా, అన్ని వెక్టార్ ఎంబెడ్డింగ్లు `vocab.vectors` మ్యాట్రిక్స్‌లో నిల్వ చేయబడ్డాయి. సాదా కాపీచేసే ప్రక్రియతో ఆ బరువులను ఎంబెడ్డింగ్ లేయర్ బరువులలో లోడ్ చేయడం చాలా సులభమవుతుంది:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

ఇప్పుడు మన మోడల్‌ను శిక్షణ అందించి మనకు మెరుగైన ఫలితాలు వస్తాయా చూసేద్దాం:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

మనం ఖచ్చితత్వంలో గణనీయమైన పెరుగుదలను చూడకపోవడానికి ఒక కారణం మన డేటాసెట్ నుండి కొన్ని పదాలు ప్రీ-ట్రెయిన్ చేసిన GloVe శబ్దకోశంలో లేదని, అందువల్ల అవి మూలంగా నిర్లక్ష్యమవుతాయని ఉంది. ఈ సత్యాన్ని అధిగమించడానికి, మన డేటాసెట్ పై మన స్వంత ఎంబెడ్డింగ్స్ శిక్షణ ఇవ్వవచ్చు. 


## సాందర్భిక ఎంపికలు

Word2Vec వంటి సంప్రదాయ ప్రీట్రెయిన్ చేసిన ఎంపికల ప్రతినిధుల ఒక ప్రధాన పరిమితి పదార్థ అర్ధ స్పష్టత సమస్య. ప్రీట్రెయిన్ చేసిన ఎంపికలు కొన్ని సందర్భాల్లో పదజాలం అర్ధాలను పట్టుకోవచ్చు కానీ, ఒక పదం యొక్క ప్రతి సంభావ్య అర్థం అదే ఎంపికలో కోడ్ చేయబడుతుంది. ఇది దిగువబడే నమూనాల్లో సమస్యలను కలిగిస్తుంది, ఎందుకంటే 'play' వంటి చాలా పదాలు ఉపయోగించిన సందర్భంపై ఆధారపడి వేరువేరుగా అర్థాలను కలిగి ఉంటాయి.

ఉదాహరణకు 'play' పదం ఆ రెండు వాక్యాలలో పూర్తిగా విభిన్న అర్థాలను కలిగి ఉంది:
- నేను థియేటర్ లో ఒక **play** కి వెళ్లాను.
- జాన్ తన స్నేహితులతో **play** చేయాలని కోరుకుంటున్నాడు.

పై pretrained ఎంపికలు 'play' పదం యొక్క ఈ రెండు అర్థాలను ఒకే ఎంపికలో ప్రాతినిధ్యం వహిస్తాయి. ఈ పరిమితిని అధిగమించడానికి, మనం పెద్ద పాఠ్య శేఖరణపై శిక్షణ పొందిన **భాషా మోడల్** ఆధారంగా ఎంపికలను నిర్మించాలి, ఇది పదాలను విభిన్న సందర్భాల్లో ఎలా ఉంచుకోవచ్చో *గుర్తిస్తుంద*. సాందర్భిక ఎంపికలపై చర్చ ఈ పాఠశాలలో చర్చా పరిధి లో కాదు, కానీ భాషా మోడల్స్ గురించి వచ్చే యూనిట్ లో మళ్ళీ మేము వాటిని మాట్లాడతాము.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
